# TICKETMELT — GRPO Training on Qwen-2.5-3B-Instruct

Train a small LLM to coordinate incident response in the TicketMelt multi-agent environment using GRPO.

In [ ]:
!pip install -q unsloth trl transformers datasets peft accelerate bitsandbytes
!pip install -q fastapi uvicorn httpx

In [ ]:
# Clone the repo — replace with your actual repo URL
!git clone https://huggingface.co/spaces/TheAllanB/ticketmelt /content/ticketmelt
import sys
sys.path.insert(0, '/content/ticketmelt')

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LEN = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'v_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('Model loaded with LoRA adapters')

In [ ]:
from src.environment import TicketmeltEnv
from src.prompt import observation_to_prompt, parse_action

def ticketmelt_reward_fn(completions, episode_seeds=None, **kwargs):
    """GRPO reward: run one episode per completion, first action = model output, rest = heuristic."""
    rewards = []
    for i, completion in enumerate(completions):
        seed = (episode_seeds[i] if episode_seeds else i)
        env = TicketmeltEnv(seed=seed)
        obs = env.reset(seed=seed)

        # Round 0: use the model's action
        action = parse_action(completion)
        obs, reward, done, info = env.step(action)

        # Remaining rounds: avoid-collision heuristic
        while not done:
            if obs.my_service.completed:
                fallback = 'MONITOR'
            elif obs.history:
                last = obs.history[-1]
                a_count = sum(1 for v in last.commitments.values() if v == 'DEPLOY_PROD_A')
                fallback = 'DEPLOY_PROD_B' if a_count >= 2 else 'DEPLOY_PROD_A'
            else:
                fallback = 'DEPLOY_PROD_B'
            from src.models import Action
            obs, reward, done, info = env.step(Action(commitment=fallback, channel_msg=''))

        rewards.append(float(reward))
    return rewards

In [ ]:
from datasets import Dataset

N_PROMPTS = 500
rows = []
for seed in range(N_PROMPTS):
    env = TicketmeltEnv(seed=seed)
    obs = env.reset(seed=seed)
    rows.append({'prompt': observation_to_prompt(obs), 'seed': seed})

dataset = Dataset.from_list(rows)
print(f'Dataset: {len(dataset)} prompts')
print(dataset[0]['prompt'][:400])

In [ ]:
from trl import GRPOConfig, GRPOTrainer

def reward_with_seeds(completions, prompts=None, **kwargs):
    seeds = kwargs.get('episode_seeds', list(range(len(completions))))
    return ticketmelt_reward_fn(completions, episode_seeds=seeds)

training_args = GRPOConfig(
    output_dir='ticketmelt_grpo_checkpoint',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    logging_steps=10,
    save_steps=100,
    max_prompt_length=MAX_SEQ_LEN - 64,
    max_completion_length=64,
    num_generations=4,
    report_to='none',
)
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=reward_with_seeds,
    train_dataset=dataset,
    tokenizer=tokenizer,
)
print('Trainer configured')

In [ ]:
import json
from src.rollout import run_episode

def quick_eval(model, tokenizer, n=20, seed_offset=1000):
    FastLanguageModel.for_inference(model)
    wins = 0
    comps = {'r1': [], 'r2': [], 'r3': [], 'r4': []}
    for i in range(n):
        env = TicketmeltEnv(seed=seed_offset + i)
        res = run_episode(model, tokenizer, env, observation_to_prompt, seed=seed_offset+i)
        wins += int(res['final_reward'] == 1.0)
        bd = res['info'].get('reward_breakdown', {})
        comps['r1'].append(bd.get('r1_service_restored', 0))
        comps['r2'].append(bd.get('r2_site_uptime', 0))
        comps['r3'].append(bd.get('r3_clean_deploys', 0))
        comps['r4'].append(bd.get('r4_yield_to_critical', 0))
    return {'win_rate': wins/n, **{k: sum(v)/len(v) for k, v in comps.items()}}

pre = quick_eval(model, tokenizer)
print(f'PRE-TRAINING  win={pre["win_rate"]:.1%} R1={pre["r1"]:.3f} R2={pre["r2"]:.3f} R3={pre["r3"]:.3f} R4={pre["r4"]:.3f}')

In [ ]:
FastLanguageModel.for_training(model)
trainer.train()
print('Training complete')

In [ ]:
post = quick_eval(model, tokenizer)
print(f'POST-TRAINING win={post["win_rate"]:.1%} R1={post["r1"]:.3f} R2={post["r2"]:.3f} R3={post["r3"]:.3f} R4={post["r4"]:.3f}')

with open('training_results.json', 'w') as f:
    json.dump({'pre': pre, 'post': post}, f, indent=2)
print('Results saved to training_results.json')

In [ ]:
model.save_pretrained('ticketmelt_final')
tokenizer.save_pretrained('ticketmelt_final')
print('Checkpoint saved to ticketmelt_final/')